# TP 1: LDA/QDA y optimización matemática de modelos

# Intro teórica

## Definición: Clasificador Bayesiano

Sean $k$ poblaciones, $x \in \mathbb{R}^p$ puede pertenecer a cualquiera $g \in \mathcal{G}$ de ellas. Bajo un esquema bayesiano, se define entonces $\pi_j \doteq P(G = j)$ la probabilidad *a priori* de que $X$ pertenezca a la clase *j*, y se **asume conocida** la distribución condicional de cada observable dado su clase $f_j \doteq f_{X|G=j}$.

De esta manera dicha probabilidad *a posteriori* resulta
$$
P(G|_{X=x} = j) = \frac{f_{X|G=j}(x) \cdot p_G(j)}{f_X(x)} \propto f_j(x) \cdot \pi_j
$$

La regla de decisión de Bayes es entonces
$$
H(x) \doteq \arg \max_{g \in \mathcal{G}} \{ P(G|_{X=x} = j) \} = \arg \max_{g \in \mathcal{G}} \{ f_j(x) \cdot \pi_j \}
$$

es decir, se predice a $x$ como perteneciente a la población $j$ cuya probabilidad a posteriori es máxima.

*Ojo, a no desesperar! $\pi_j$ no es otra cosa que una constante prefijada, y $f_j$ es, en su esencia, un campo escalar de $x$ a simplemente evaluar.*

## Distribución condicional

Para los clasificadores de discriminante cuadrático y lineal (QDA/LDA) se asume que $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma_j)$, es decir, se asume que cada población sigue una distribución normal.

Por definición, se tiene entonces que para una clase $j$:
$$
f_j(x) = \frac{1}{(2 \pi)^\frac{p}{2} \cdot |\Sigma_j|^\frac{1}{2}} e^{- \frac{1}{2}(x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j)}
$$

Aplicando logaritmo (que al ser una función estrictamente creciente no afecta el cálculo de máximos/mínimos), queda algo mucho más práctico de trabajar:

$$
\log{f_j(x)} = -\frac{1}{2}\log |\Sigma_j| - \frac{1}{2} (x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j) + C
$$

Observar que en este caso $C=-\frac{p}{2} \log(2\pi)$, pero no se tiene en cuenta ya que al tener una constante aditiva en todas las clases, no afecta al cálculo del máximo.

## LDA

En el caso de LDA se hace una suposición extra, que es $X|_{G=j} \sim \mathcal{N}_p(\mu_j, \Sigma)$, es decir que las poblaciones no sólo siguen una distribución normal sino que son de igual matriz de covarianzas. Reemplazando arriba se obtiene entonces:

$$
\log{f_j(x)} =  -\frac{1}{2}\log |\Sigma| - \frac{1}{2} (x-\mu_j)^T \Sigma^{-1} (x- \mu_j) + C
$$

Ahora, como $-\frac{1}{2}\log |\Sigma|$ es común a todas las clases se puede incorporar a la constante aditiva y, distribuyendo y reagrupando términos sobre $(x-\mu_j)^T \Sigma^{-1} (x- \mu_j)$ se obtiene finalmente:

$$
\log{f_j(x)} =  \mu_j^T \Sigma^{-1} (x- \frac{1}{2} \mu_j) + C'
$$

## Entrenamiento/Ajuste

Obsérvese que para ambos modelos, ajustarlos a los datos implica estimar los parámetros $(\mu_j, \Sigma_j) \; \forall j = 1, \dots, k$ en el caso de QDA, y $(\mu_j, \Sigma)$ para LDA.

Estos parámetros se estiman por máxima verosimilitud, de manera que los estimadores resultan:

* $\hat{\mu}_j = \bar{x}_j$ el promedio de los $x$ de la clase *j*
* $\hat{\Sigma}_j = s^2_j$ la matriz de covarianzas estimada para cada clase *j*
* $\hat{\pi}_j = f_{R_j} = \frac{n_j}{n}$ la frecuencia relativa de la clase *j* en la muestra
* $\hat{\Sigma} = \frac{1}{n} \sum_{j=1}^k n_j \cdot s^2_j$ el promedio ponderado (por frecs. relativas) de las matrices de covarianzas de todas las clases. *Observar que se utiliza el estimador de MV y no el insesgado*

Es importante notar que si bien todos los $\mu, \Sigma$ deben ser estimados, la distribución *a priori* puede no inferirse de los datos sino asumirse previamente, utilizándose como entrada del modelo.

## Predicción

Para estos modelos, al igual que para cualquier clasificador Bayesiano del tipo antes visto, la estimación de la clase es por método *plug-in* sobre la regla de decisión $H(x)$, es decir devolver la clase que maximiza $\hat{f}_j(x) \cdot \hat{\pi}_j$, o lo que es lo mismo $\log\hat{f}_j(x) + \log\hat{\pi}_j$.

# Código provisto

Con el fin de no retrasar al alumno con cuestiones estructurales y/o secundarias al tema que se pretende tratar, se provee una base de código que **no es obligatoria de usar** pero se asume que resulta resulta beneficiosa.

In [195]:
import numpy as np
import pandas as pd
import numpy.linalg as LA
from scipy.linalg import cholesky, solve_triangular
from scipy.linalg.lapack import dtrtri

## Base code

In [196]:
class BaseBayesianClassifier:
  def __init__(self):
    pass

  def _estimate_a_priori(self, y):
    a_priori = np.bincount(y.flatten().astype(int)) / y.size
    # Q3: para que sirve bincount?
    # bincount cuenta cuántas veces aparece cada clase en y. Al dividir por y.size
    # se obtiene la frecuencia relativa de cada clase, es decir, una estimación de P(G=j).
    # Ejemplo: si y = [0,1,1,0,2,1] => bincount = [2,3,1] => a_priori = [2/6, 3/6, 1/6].
    # Se retorna en log (log-prior) porque el clasificador trabaja en log-space
    # para evitar underflow numérico al multiplicar probabilidades pequeñas.
    return np.log(a_priori)

  def _fit_params(self, X, y):
    # estimate all needed parameters for given model
    raise NotImplementedError()

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    raise NotImplementedError()

  def fit(self, X, y, a_priori=None):
    # if it's needed, estimate a priori probabilities
    self.log_a_priori = self._estimate_a_priori(y) if a_priori is None else np.log(a_priori)

    # now that everything else is in place, estimate all needed parameters for given model
    self._fit_params(X, y)
    # Q4: por que el _fit_params va al final? no se puede mover a, por ejemplo, antes de la priori?
    #
    # La distribución a priori (π_j = P(G=j)) es la probabilidad de pertenecer a cada clase
    # ANTES de observar los features. Es una decisión de modelado que define la estructura
    # del clasificador: cuántas clases considerar y con qué peso previo. Puede estimarse
    # de los datos (bincount) o asumirse externamente (parámetro a_priori).
    #
    # Los parámetros condicionales (μ_k, Σ_k) se estiman DENTRO de esa estructura.
    # Siguiendo la lógica bayesiana:
    #   1) Primero se define el prior → estructura del modelo (cuántas clases, con qué pesos).
    #   2) Luego se estiman los parámetros condicionales de cada clase (μ_k, Σ_k).
    #
    # Por eso _fit_params no puede ir antes: el prior define el espacio del modelo,
    # y los parámetros condicionales se ajustan dentro de ese espacio.
    # (En el código, _fit_params itera con range(len(self.log_a_priori)) para saber
    # cuántas clases ajustar — si se moviera antes, self.log_a_priori no existiría aún.)

  def predict(self, X):
    # this is actually an individual prediction encased in a for-loop
    m_obs = X.shape[1]
    y_hat = np.empty(m_obs, dtype=int)

    for i in range(m_obs):
      y_hat[i] = self._predict_one(X[:,i].reshape(-1,1))

    # return prediction as a row vector (matching y)
    return y_hat.reshape(1,-1)

  def _predict_one(self, x):
    # calculate all log posteriori probabilities (actually, +C)
    log_posteriori = [ log_a_priori_i + self._predict_log_conditional(x, idx) for idx, log_a_priori_i
                  in enumerate(self.log_a_priori) ]

    # return the class that has maximum a posteriori probability
    return np.argmax(log_posteriori)

In [197]:
class QDA(BaseBayesianClassifier):

  def _fit_params(self, X, y):
    # estimate each covariance matrix
    self.inv_covs = [LA.inv(np.cov(X[:,y.flatten()==idx], bias=True))
                      for idx in range(len(self.log_a_priori))]
    # Q5: por que hace falta el flatten y no se puede directamente X[:,y==idx]?
    # y tiene shape (n, 1) (vector columna). y==idx da una máscara 2D de shape (n, 1),
    # y X[:, máscara_2D] no selecciona columnas correctamente (colapsa dimensiones).
    # y.flatten()==idx da una máscara 1D de shape (n,), que sí funciona como indexing
    # por columnas: X[:, máscara_1D] devuelve la submatriz (p, n_k) esperada.

    # Q6: por que se usa bias=True en vez del default bias=False?
    # bias=False divide por (N-1): estimador insesgado de la covarianza.
    # bias=True divide por N: estimador de máxima verosimilitud (MLE).
    # Se usa bias=True porque el modelo estima todos sus parámetros por MLE
    # (las medias con .mean() y el prior con frecuencias relativas también son MLE),
    # y el MLE de la covarianza de una gaussiana divide por N, no por N-1.

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]
    # Q7: que hace axis=1? por que no axis=0?
    # X tiene shape (p, n): features en filas, observaciones en columnas.
    # X[:,y.flatten()==idx] da shape (p, n_k) — las observaciones de la clase k.
    # axis=1 promedia sobre columnas (observaciones) → vector de largo p = media μ_k.
    # axis=0 promediaría sobre filas (features) → vector de largo n_k, sin sentido.
    # keepdims=True mantiene shape (p, 1) para que x - μ_k funcione por broadcasting.

  def _predict_log_conditional(self, x, class_idx):
    # Calcula log f_j(x): el log de la densidad gaussiana de x dada la clase j.
    # Viene de log N(x|μ_j, Σ_j) = -½ log|Σ_j| - ½(x-μ_j)ᵀ Σ_j⁻¹ (x-μ_j) + cte.
    # La constante -(p/2)log(2π) se omite porque es igual para todas las clases
    # y no afecta el argmax en _predict_one.
    inv_cov = self.inv_covs[class_idx]          # Σ_j⁻¹
    unbiased_x =  x - self.means[class_idx]     # (x - μ_j)
    # +½ log|Σ_j⁻¹| (= -½ log|Σ_j|, penaliza clases dispersas) - ½ dist. Mahalanobis²
    return 0.5*np.log(LA.det(inv_cov)) -0.5 * unbiased_x.T @ inv_cov @ unbiased_x

In [198]:
class TensorizedQDA(QDA):

    def _fit_params(self, X, y):
        # ask plain QDA to fit params
        super()._fit_params(X,y)

        # stack apila las listas de matrices en un solo tensor con una nueva dimensión
        # para las clases, permitiendo operar sobre todas las clases a la vez
        # en vez de iterar con un for-loop clase por clase.
        self.tensor_inv_cov = np.stack(self.inv_covs)  # lista de k (p,p) → tensor (k, p, p)
        self.tensor_means = np.stack(self.means)        # lista de k (p,1) → tensor (k, p, 1)

        # print(f'tensor_inv_cov shape: {self.tensor_inv_cov.shape}')  # (k, p, p)
        # print(f'tensor_means shape:   {self.tensor_means.shape}')    # (k, p, 1)

    def _predict_log_conditionals(self,x):
        # Calcula log f_j(x) para TODAS las clases simultáneamente (sin for-loop).
        unbiased_x = x - self.tensor_means              # (k, p, 1) por broadcasting
        inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x  # (k,1,1) dist. Mahalanobis² por clase

        return 0.5*np.log(LA.det(self.tensor_inv_cov)) - 0.5 * inner_prod.flatten()  # vector (k,) con log f_j(x) por clase

    def _predict_one(self, x):
        # return the class that has maximum a posteriori probability
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

In [199]:
class QDA_Chol1(BaseBayesianClassifier):
  # En vez de invertir Σ directamente (como QDA), usa la descomposición de Cholesky:
  # Σ = L Lᵀ, y guarda L⁻¹ (calculada con LA.inv, inversión genérica).

  def _fit_params(self, X, y):
    self.L_invs = [
        LA.inv(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True))
        # cholesky: Σ_j = L_j L_jᵀ  →  LA.inv: calcula L_j⁻¹ con inversión genérica
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    # y = L⁻¹(x - μ), entonces yᵀy = (x-μ)ᵀ (L⁻¹)ᵀ L⁻¹ (x-μ) = (x-μ)ᵀ Σ⁻¹ (x-μ)
    y = L_inv @ unbiased_x

    # log|L⁻¹ diagonal| = log|L⁻¹| = -log|L| = -½ log|Σ|
    # (y**2).sum() = distancia de Mahalanobis²
    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

In [200]:
class QDA_Chol2(BaseBayesianClassifier):
  # A diferencia de Chol1, NO invierte L. Guarda L directamente y resuelve
  # el sistema triangular Ly = (x-μ) con solve_triangular (más eficiente que invertir).

  def _fit_params(self, X, y):
    self.Ls = [
        cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True)
        # Solo calcula L_j (sin invertir) → menos costo en fit
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L = self.Ls[class_idx]
    unbiased_x =  x - self.means[class_idx]

    # Resuelve Ly = (x-μ) sin calcular L⁻¹ explícitamente (forward substitution)
    y = solve_triangular(L, unbiased_x, lower=True)

    # -log|L diagonal| = -log|L| = -½ log|Σ|  (signo negativo porque usa L, no L⁻¹)
    # (y**2).sum() = distancia de Mahalanobis²
    return -np.log(L.diagonal().prod()) -0.5 * (y**2).sum()

In [201]:
class QDA_Chol3(BaseBayesianClassifier):
  # Como Chol1 guarda L⁻¹, pero en vez de usar LA.inv (inversión genérica),
  # usa dtrtri (LAPACK) que es una inversión especializada para matrices triangulares
  # → más rápida y numéricamente estable.

  def _fit_params(self, X, y):
    self.L_invs = [
        dtrtri(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True), lower=1)[0]
        # cholesky: Σ_j = L_j L_jᵀ  →  dtrtri: calcula L_j⁻¹ explotando que L es triangular
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    # y = L⁻¹(x - μ), entonces yᵀy = (x-μ)ᵀ Σ⁻¹ (x-μ)
    y = L_inv @ unbiased_x

    # Mismo cálculo que Chol1, la diferencia es cómo se obtuvo L⁻¹ (dtrtri vs LA.inv)
    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

## Datasets

Observar que se proveen **4 datasets diferentes**, el código de ejemplo usa uno solo pero eso no significa que ustedes se limiten al mismo. También pueden usar otros datasets de su elección.

In [202]:
from sklearn.datasets import load_iris, fetch_openml, load_wine
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

def get_iris_dataset():
  data = load_iris()
  X_full = data.data
  y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
  return X_full, y_full

def get_penguins_dataset():
    # get data
    df, tgt = fetch_openml(name="penguins", return_X_y=True, as_frame=True, parser='auto')

    # drop non-numeric columns
    df.drop(columns=["island","sex"], inplace=True)

    # drop rows with missing values
    mask = df.isna().sum(axis=1) == 0
    df = df[mask]
    tgt = tgt[mask]

    return df.values, tgt.to_numpy().reshape(-1,1)

def get_wine_dataset():
    # get data
    data = load_wine()
    X_full = data.data
    y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
    return X_full, y_full

def get_letters_dataset():
    # get data
    letter = fetch_openml('letter', version=1, as_frame=False)
    return letter.data, letter.target.reshape(-1,1)

def label_encode(y_full):
    # Convierte etiquetas de texto a enteros (ej: ["class_0","class_1"] → [0, 1]).
    # flatten: LabelEncoder solo acepta 1D. reshape: vuelve al shape original.
    return LabelEncoder().fit_transform(y_full.flatten()).reshape(y_full.shape)

def split_transpose(X, y, test_size, random_state):
    # X_train, X_test, y_train, y_test but all transposed
    return [elem.T for elem in train_test_split(X, y, test_size=test_size, random_state=random_state)]

## Benchmarking

Nota: esta clase fue creada bastante rápido y no pretende ser una plataforma súper confiable sobre la que basarse, sino más bien una herramienta simple con la que poder medir varios runs y agregar la información.

En forma rápida, `warmup` es la cantidad de runs para warmup, `mem_runs` es la cantidad de runs en las que se mide el pico de uso de RAM y `n_runs` es la cantidad de runs en las que se miden tiempos.

La razón por la que se separan es que medir memoria hace ~2.5x más lento cada run, pero al mismo tiempo se estabiliza mucho más rápido.

**Importante:** tener en cuenta que los modelos que predicen en batch (usan `predict` directamente) deberían consumir, como mínimo, $n$ veces la memoria de los que predicen por observación.

In [203]:
import time
from tqdm.notebook import tqdm
from numpy.random import RandomState
import tracemalloc

RNG_SEED = 6553

class Benchmark:
    def __init__(self, X, y, n_runs=1000, warmup=100, mem_runs=100, test_sz=0.3, rng_seed=RNG_SEED, same_splits=True):
        self.X = X
        self.y = y
        self.n = n_runs
        self.warmup = warmup
        self.mem_runs = mem_runs
        self.test_sz = test_sz
        self.det = same_splits
        if self.det:
            self.rng_seed = rng_seed
        else:
            self.rng = RandomState(rng_seed)

        self.data = dict()

        print("Benching params:")
        print("Total runs:",self.warmup+self.mem_runs+self.n)
        print("Warmup runs:",self.warmup)
        print("Peak Memory usage runs:", self.mem_runs)
        print("Running time runs:", self.n)
        approx_test_sz = int(self.y.size * self.test_sz)
        print("Train size rows (approx):",self.y.size - approx_test_sz)
        print("Test size rows (approx):",approx_test_sz)
        print("Test size fraction:",self.test_sz)

    def bench(self, model_class, **kwargs):
        name = model_class.__name__
        time_data = np.empty((self.n, 3), dtype=float)  # train_time, test_time, accuracy
        mem_data = np.empty((self.mem_runs, 2), dtype=float)  # train_peak_mem, test_peak_mem
        rng = RandomState(self.rng_seed) if self.det else self.rng


        for i in range(self.warmup):
            # Instantiate model with error check for unsupported parameters
            model = model_class(**kwargs)

            # Generate current train-test split
            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )
            # Run training and prediction (timing or memory measurement not recorded)
            model.fit(X_train, y_train)
            model.predict(X_test)

        for i in tqdm(range(self.mem_runs), total=self.mem_runs, desc=f"{name} (MEM)"):

            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            tracemalloc.start()

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()

            _, train_peak = tracemalloc.get_traced_memory()
            tracemalloc.reset_peak()

            model.predict(X_test)
            t3 = time.perf_counter()
            _, test_peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            mem_data[i,] = (
                train_peak / (1024 * 1024),
                test_peak / (1024 * 1024)
            )

        for i in tqdm(range(self.n), total=self.n, desc=f"{name} (TIME)"):
            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()
            preds = model.predict(X_test)
            t3 = time.perf_counter()

            time_data[i,] = (
                (t2 - t1) * 1000,
                (t3 - t2) * 1000,
                (y_test.flatten() == preds.flatten()).mean()
            )

        self.data[name] = (time_data, mem_data)

    def summary(self, baseline=None):
        aux = []
        for name, (time_data, mem_data) in self.data.items():
            result = {
                'model': name,
                'train_median_ms': np.median(time_data[:, 0]),
                'train_std_ms': time_data[:, 0].std(),
                'test_median_ms': np.median(time_data[:, 1]),
                'test_std_ms': time_data[:, 1].std(),
                'mean_accuracy': time_data[:, 2].mean(),
                'train_mem_median_mb': np.median(mem_data[:, 0]),
                'train_mem_std_mb': mem_data[:, 0].std(),
                'test_mem_median_mb': np.median(mem_data[:, 1]),
                'test_mem_std_mb': mem_data[:, 1].std()
            }
            aux.append(result)
        df = pd.DataFrame(aux).set_index('model')

        if baseline is not None and baseline in self.data:
            df['train_speedup'] = df.loc[baseline, 'train_median_ms'] / df['train_median_ms']
            df['test_speedup'] = df.loc[baseline, 'test_median_ms'] / df['test_median_ms']
            df['train_mem_reduction'] = df.loc[baseline, 'train_mem_median_mb'] / df['train_mem_median_mb']
            df['test_mem_reduction'] = df.loc[baseline, 'test_mem_median_mb'] / df['test_mem_median_mb']
        return df

## Ejemplo

In [204]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

X_full.shape, y_full.shape

((178, 13), (178, 1))

In [205]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [206]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [207]:
# bencheamos un par
to_bench = [QDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [208]:
# como es una clase, podemos seguir bencheando más después
b.bench(TensorizedQDA)

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [209]:
# hacemos un summary
b.summary()

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.217979,0.259851,1.418584,2.430690,0.982407,0.018593,0.000690,0.007748,0.000348
TensorizedQDA,0.184833,0.325695,0.681354,1.455727,0.982593,0.018593,0.000652,0.012199,0.000316


In [210]:
# son muchos datos! nos quedamos con un par nomás
summ = b.summary()

# como es un pandas DataFrame, subseteamos columnas fácil
summ[['train_median_ms', 'test_median_ms','mean_accuracy']]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
QDA,0.217979,1.418584,0.982407
TensorizedQDA,0.184833,0.681354,0.982593


In [211]:
# podemos setear un baseline para que fabrique columnas de comparación
summ = b.summary(baseline='QDA')

summ

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.217979,0.259851,1.418584,2.430690,0.982407,0.018593,0.000690,0.007748,0.000348,1.000000,1.000000,1.0,1.000000
TensorizedQDA,0.184833,0.325695,0.681354,1.455727,0.982593,0.018593,0.000652,0.012199,0.000316,1.179332,2.082005,1.0,0.635124


In [212]:
summ[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.217979,1.418584,0.982407,1.000000,1.000000,1.0,1.000000
TensorizedQDA,0.184833,0.681354,0.982593,1.179332,2.082005,1.0,0.635124


# Consigna QDA

**Notación**: en general notamos

* $k$ la cantidad de clases
* $n$ la cantidad de observaciones
* $p$ la cantidad de features/variables/predictores

**Sugerencia:** combinaciones adecuadas de `transpose`, `stack`, `reshape` y, ocasionalmente, `flatten` y `diagonal` suele ser más que suficiente. Se recomienda *fuertemente* explorar la dimensionalidad de cada elemento antes de implementar las clases.

## Tensorización

En esta sección nos vamos a ocupar de hacer que el modelo sea más rápido para generar predicciones, observando que incurre en un doble `for` dado que predice en forma individual un escalar para cada observación, para cada clase. Paralelizar ambos vía tensorización suena como una gran vía de mejora de tiempos.

### 1) Diferencias entre `QDA`y `TensorizedQDA`

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?

Paraleliza sobre las $k$ clases, porque se puede ver que el `predict` original de la clase base `BaseBayesianClassifier` aun mantiene el for loop sobre las $n$ observaciones. Solo se eliminó el for-loop interno sobre las $k$ clases.

2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

Los shapes de los tensores son:

- `tensor_inv_cov = np.stack(inv_covs)` $\rightarrow (k, p, p)$: las $k$ matrices de precisión $\Sigma_j^{-1}$ (inversas de las matrices de covarianza) apiladas en un solo tensor.
- `tensor_means = np.stack(means)` $\rightarrow (k, p, 1)$: los $k$ vectores de medias $\mu_j$ apilados.
- $x$ tiene shape $(p, 1)$: una sola observación.

Recordar que $\Sigma_j$ es la matriz de covarianza de la clase $j$ (describe la dispersión y correlación de los features), y $\Sigma_j^{-1}$ es su inversa, conocida como **matriz de precisión**. En el código, QDA guarda directamente $\Sigma_j^{-1}$ porque es lo que necesita la fórmula del log-condicional.

El método `_predict_log_conditionals(x)` opera paso a paso así:

**1)** `unbiased_x = x - self.tensor_means` $\rightarrow (p, 1) - (k, p, 1) = (k, p, 1)$

Por broadcasting, NumPy expande $x$ y lo resta a cada $\mu_j$. Se obtienen los $k$ vectores $(x - \mu_j)$ apilados.

**2)** `unbiased_x.transpose(0,2,1)` $\rightarrow (k, p, 1) \to (k, 1, p)$

Transpone dentro de cada "rodaja" (ejes 1 y 2), convirtiendo cada vector columna en vector fila. Equivale al `.T` que hace QDA.

**3)** `... @ self.tensor_inv_cov @ unbiased_x` $\rightarrow (k, 1, p) \times (k, p, p) \times (k, p, 1) = (k, 1, 1)$

NumPy hace matmul por rodaja: para cada clase $j$ calcula $(x - \mu_j)^T \Sigma_j^{-1} (x - \mu_j)$, que es la **distancia de Mahalanobis al cuadrado**. Esta distancia mide qué tan lejos está $x$ del centro $\mu_j$ de la clase, ponderado por la forma de la distribución ($\Sigma_j$): una clase muy dispersa en cierta dirección "penaliza menos" en esa dirección. Es lo mismo que hace QDA en `_predict_log_conditional`, pero para las $k$ clases en paralelo.

**4)** `inner_prod.flatten()` $\rightarrow (k, 1, 1) \to (k,)$: un escalar por clase.

**5)** `0.5*np.log(LA.det(self.tensor_inv_cov))` $\rightarrow (k,)$

`LA.det` sobre un tensor $(k, p, p)$ calcula un determinante por clase. Se obtiene $+\frac{1}{2} \log |\Sigma_j^{-1}|$ para cada $j$. Este término penaliza clases con covarianza grande (más dispersas): a mayor $|\Sigma_j|$, menor $|\Sigma_j^{-1}|$ y más negativo el log.

**6)** El return resta ambos vectores $(k,)$, dando $\log f_j(x)$ para cada clase en un solo vector.

Cada entrada de ese vector es numéricamente idéntica a lo que QDA calcula en `_predict_log_conditional` para una clase individual. La diferencia es que QDA lo hace dentro de un for-loop (una clase por iteración) mientras que `TensorizedQDA` obtiene el vector completo de las $k$ clases en una sola operación tensorizada.

Con ese vector $(k,)$ de log-condicionales, solo queda obtener el máximo a posteriori en `_predict_one`:

```python
return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))
```

Suma el vector de log-priors $(k,)$ con el vector de log-condicionales $(k,)$, obteniendo las log-probabilidades a posteriori de cada clase, y `argmax` devuelve la clase ganadora.

#### Intuición: ¿qué significa que una clase dispersa "penalice menos"?

Supongamos dos clases en 2D:

- **Clase A**: covarianza "redonda" (dispersión igual en todas las direcciones)
- **Clase B**: covarianza "alargada" (muy dispersa en la dirección horizontal, concentrada en la vertical)

```
Clase A (redonda)          Clase B (alargada)
      · · ·                · · · · · · · · ·
    · · · · ·              · · · · · · · · ·
    · · μ · ·              · · · · μ · · · ·
    · · · · ·              · · · · · · · · ·
      · · ·                · · · · · · · · ·
```

Si llega un punto $x$ que está lejos **horizontalmente** de ambos centros:

- Para la **Clase A**, esa distancia pesa mucho $\rightarrow$ Mahalanobis alta $\rightarrow$ "poco probable"
- Para la **Clase B**, esa distancia pesa poco, porque la clase ya es muy dispersa en esa dirección $\rightarrow$ Mahalanobis baja $\rightarrow$ "razonable"

Esto es lo que hace $\Sigma_j^{-1}$ en la distancia de Mahalanobis: **reescala las distancias según la dispersión de cada clase en cada dirección**. En las direcciones donde la clase tiene mucha varianza, la inversa $\Sigma^{-1}$ tiene valores chicos, así que esa componente contribuye poco a la distancia. Es como decir: "si la clase ya se expande mucho para ese lado, no sorprende encontrar un punto lejos en esa dirección".

La distancia euclidiana trataría ambas direcciones igual. La de Mahalanobis las pondera por la forma de la nube de puntos de cada clase.

#### Intuición: ¿por qué hace falta calcular el determinante?

La distancia de Mahalanobis mide **qué tan lejos** está $x$ del centro de la clase. Pero eso solo no alcanza.

El determinante $|\Sigma_j|$ mide el **volumen** de la nube de puntos de la clase. Una clase con covarianza grande tiene un determinante grande, lo que significa que "reparte" su probabilidad en un espacio más amplio $\rightarrow$ la densidad en cada punto es más baja.

Usando el ejemplo anterior: un punto $x$ que está **justo en el centro** de ambas clases ($x = \mu_j$) tiene Mahalanobis = 0 para las dos.

```
Clase A (compacta)         Clase B (dispersa)
      · · ·                · · · · · · · · ·
    · · · · ·              · · · · · · · · ·
    · · μ · ·              · · · · μ · · · ·
    · · · · ·              · · · · · · · · ·
      · · ·                · · · · · · · · ·
```

Sin el determinante, ambas clases tendrían el mismo score. Pero la Clase A, al ser más compacta, concentra más densidad de probabilidad en su centro que la Clase B. El término $-\frac{1}{2}\log|\Sigma_j|$ captura eso: penaliza clases más dispersas (determinante grande $\rightarrow$ log grande $\rightarrow$ score más negativo).

En el código se calcula como `0.5*np.log(LA.det(tensor_inv_cov))`, que es $+\frac{1}{2}\log|\Sigma_j^{-1}| = -\frac{1}{2}\log|\Sigma_j|$ (misma cosa, signo invertido porque usa la inversa).

### 2) Optimización

Debido a la forma cuadrática de QDA, no se puede predecir para $n$ observaciones en una sola pasada (utilizar $X \in \mathbb{R}^{p \times n}$ en vez de $x \in \mathbb{R}^p$) sin pasar por una matriz de $n \times n$ en donde se computan todas las interacciones entre observaciones. Se puede acceder al resultado recuperando sólo la diagonal de dicha matriz, pero resulta ineficiente en tiempo y (especialmente) en memoria. Aún así, es *posible* que el modelo funcione más rápido.

4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.

Aparece en la forma cuadrática al intentar predecir todas las $n$ observaciones de golpe. En `_predict_log_conditional`, la distancia de Mahalanobis para una observación es:

$$
(x - \mu_j)^T \Sigma_j^{-1} (x - \mu_j) \rightarrow (1, p) \times (p, p) \times (p, 1) = \text{escalar}
$$

Si en vez de una sola $x$ usamos todas las observaciones $X \in \mathbb{R}^{p \times n}$:

$$
U = X - \mu_j \quad \rightarrow (p, n)
$$
$$
U^T \Sigma_j^{-1} U \rightarrow (n, p) \times (p, p) \times (p, n) = (n, n)
$$

Ahí está la matriz $n \times n$. Solo la **diagonal** contiene lo que necesitamos. ¿Por qué? Porque el elemento $(i, l)$ de esa matriz es:

$$(x_i - \mu_j)^T \Sigma_j^{-1} (x_l - \mu_j)$$

- Cuando $i = l$ (diagonal): $(x_i - \mu_j)^T \Sigma_j^{-1} (x_i - \mu_j)$ $\rightarrow$ distancia de Mahalanobis² de la observación $i$ al centro de la clase. **Esto es lo que necesitamos para clasificar.**
- Cuando $i \neq l$ (fuera de diagonal): $(x_i - \mu_j)^T \Sigma_j^{-1} (x_l - \mu_j)$ $\rightarrow$ una "similitud" entre dos observaciones distintas ponderada por la estructura de la clase. **No sirve para clasificar**, porque para predecir la clase de $x_i$ solo necesitamos saber qué tan lejos está $x_i$ del centro, no cómo se relaciona con $x_l$.

En el for-loop original, solo se computa un escalar por observación (la diagonal). Al hacer batch con la matriz completa, NumPy no tiene forma de calcular solo la diagonal — hace el producto completo $n \times n$ y después se descarta todo menos la diagonal.

Esto es ineficiente en:
- **Memoria**: se aloca una matriz $n \times n$ completa ($O(n^2)$ en vez de $O(n)$)
- **Tiempo**: se calculan $n^2$ entradas cuando solo se necesitan $n$

Aun así, puede ser más rápido que el for-loop de Python porque la multiplicación matricial corre en BLAS/LAPACK optimizado, incluso desperdiciando cómputo.

3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.

In [213]:
# FasterQDA elimina el for-loop sobre observaciones de predict.
# El flujo es:
#   1. Centrar: U = X - μ_j → (k, p, n) por broadcasting
#   2. Producto: Σ_j⁻¹ U → (k, p, n)
#   3. Forma cuadrática: Uᵀ Σ⁻¹ U → (k, n, n) — acá aparece la matriz n×n
#   4. Diagonal: se extrae con np.diagonal → (k, n) — solo las distancias de Mahalanobis²
#   5. Log-posterior: suma log-prior + log-determinante - ½ Mahalanobis → (k, n)
#   6. Argmax: por columna (axis=0) → clase ganadora para cada observación

class FasterQDA(TensorizedQDA):

    def predict(self, X):
        # X shape: (p, n), tensor_means shape: (k, p, 1), tensor_inv_cov shape: (k, p, p)

        # 1. Centrar X respecto a cada clase: (p, n) - (k, p, 1) → (k, p, n)
        U = X - self.tensor_means

        # 2. Producto Σ_j⁻¹ @ U para cada clase: (k, p, p) @ (k, p, n) → (k, p, n)
        SU = self.tensor_inv_cov @ U

        # 3. Forma cuadrática Uᵀ Σ⁻¹ U por clase: (k, n, p) @ (k, p, n) → (k, n, n)
        quad = U.transpose(0, 2, 1) @ SU

        # 4. Extraer diagonal (dist. Mahalanobis² de cada obs): (k, n, n) → (k, n)
        mahal = np.diagonal(quad, axis1=1, axis2=2)

        # 5. Log-posterior por clase: (k, 1) - (k, n) → (k, n)
        log_det = 0.5 * np.log(LA.det(self.tensor_inv_cov))
        log_post = (self.log_a_priori + log_det).reshape(-1, 1) - 0.5 * mahal

        # 6. Clase con máximo a posteriori para cada obs: (n,) → (1, n)
        return np.argmax(log_post, axis=0).reshape(1, -1)

5. Demostrar que
$$
diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)
$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente,
$$
np.sum(A^T \odot B, axis=0).T
$$
queda a preferencia del alumno cuál usar.

#### Demostración

Sean $A \in \mathbb{R}^{n \times p}$ y $B \in \mathbb{R}^{p \times n}$. El producto $A \cdot B$ da una matriz de $(n \times n)$.

El elemento $(i, j)$ de $A \cdot B$ es:

$$
(A \cdot B)_{ij} = \sum_{k=1}^{p} A_{ik} \cdot B_{kj}
$$

La diagonal es el caso particular $i = j$:

$$
(A \cdot B)_{ii} = \sum_{k=1}^{p} A_{ik} \cdot B_{ki}
$$

Por otro lado, $B^T \in \mathbb{R}^{n \times p}$ y su elemento $(i, k)$ es $(B^T)_{ik} = B_{ki}$.

El producto de Hadamard (element-wise) $A \odot B^T$ tiene como elemento $(i, k)$:

$$
(A \odot B^T)_{ik} = A_{ik} \cdot (B^T)_{ik} = A_{ik} \cdot B_{ki}
$$

Sumando sobre las columnas (axis=1):

$$
\sum_{k=1}^{p} (A \odot B^T)_{ik} = \sum_{k=1}^{p} A_{ik} \cdot B_{ki} = (A \cdot B)_{ii}
$$

Que es exactamente el elemento $i$-ésimo de la diagonal. Por lo tanto:

$$
\text{diag}(A \cdot B) = \text{np.sum}(A \odot B^T, \text{axis}=1) \quad
$$

La ventaja es que $A \odot B^T$ tiene shape $(n, p)$ — nunca se construye la matriz $(n, n)$. Se pasa de $O(n^2 p)$ operaciones y $O(n^2)$ memoria a $O(np)$ en ambos casos.

#### Demostración de la forma equivalente: $\text{np.sum}(A^T \odot B, \text{axis}=0)^T$

Ahora $A^T \in \mathbb{R}^{p \times n}$ y $B \in \mathbb{R}^{p \times n}$, así que $A^T \odot B$ tiene shape $(p, n)$.

El elemento $(k, i)$ de $A^T \odot B$ es:

$$
(A^T \odot B)_{ki} = (A^T)_{ki} \cdot B_{ki} = A_{ik} \cdot B_{ki}
$$

Sumando sobre las filas (axis=0):

$$
\sum_{k=1}^{p} (A^T \odot B)_{ki} = \sum_{k=1}^{p} A_{ik} \cdot B_{ki} = (A \cdot B)_{ii}
$$

Esa suma da un vector fila de shape $(n,)$. El $.T$ al final es para mantener consistencia de shape (vector columna). El resultado es el mismo:

$$
\text{diag}(A \cdot B) = \text{np.sum}(A^T \odot B, \text{axis}=0)^T \quad 
$$

Ambas formas evitan construir la matriz $(n, n)$ y tienen costo $O(np)$. La diferencia es solo si se trabaja con las matrices en su forma original $(p, n)$ o transpuesta $(n, p)$.

#### Intuición visual

Supongamos $A$ de $(3, 2)$ y $B$ de $(2, 3)$ (es decir, $n=3$, $p=2$).

Las matrices son:

```
A (3×2)          B (2×3)
┌         ┐      ┌             ┐
│ a₁₁ a₁₂ │      │ b₁₁ b₁₂ b₁₃ │
│ a₂₁ a₂₂ │      │ b₂₁ b₂₂ b₂₃ │
│ a₃₁ a₃₂ │      └             ┘
└         ┘
```

El producto $A \cdot B$ da una matriz $3 \times 3$. Expandiendo con subíndices:

```
A · B (3×3) =
┌                                                             ┐
│ [a₁₁·b₁₁ + a₁₂·b₂₁]  a₁₁·b₁₂ + a₁₂·b₂₂   a₁₁·b₁₃ + a₁₂·b₂₃  │
│  a₂₁·b₁₁ + a₂₂·b₂₁  [a₂₁·b₁₂ + a₂₂·b₂₂]  a₂₁·b₁₃ + a₂₂·b₂₃  │
│  a₃₁·b₁₁ + a₃₂·b₂₁   a₃₁·b₁₂ + a₃₂·b₂₂  [a₃₁·b₁₃ + a₃₂·b₂₃] │
└                                                             ┘
Solo nos importan los [] (la diagonal). El resto es desperdicio.
```

Ahora con Hadamard. Transponemos $B$ para alinear los subíndices:

```
A (3×2)          B^T (3×2)
┌         ┐      ┌         ┐
│ a₁₁ a₁₂ │      │ b₁₁ b₂₁ │
│ a₂₁ a₂₂ │      │ b₁₂ b₂₂ │
│ a₃₁ a₃₂ │      │ b₁₃ b₂₃ │
└         ┘      └         ┘
```

Multiplicamos element-wise y sumamos por fila:

```
    A ⊙ B^T (3×2)            sum(axis=1)                 diag(A·B)
┌                  ┐                              ┌                   ┐
│ a₁₁·b₁₁  a₁₂·b₂₁ │    → a₁₁·b₁₁ + a₁₂·b₂₁  →    │ a₁₁·b₁₁ + a₁₂·b₂₁ │
│ a₂₁·b₁₂  a₂₂·b₂₂ │    → a₂₁·b₁₂ + a₂₂·b₂₂  →    │ a₂₁·b₁₂ + a₂₂·b₂₂ │
│ a₃₁·b₁₃  a₃₂·b₂₃ │    → a₃₁·b₁₃ + a₃₂·b₂₃  →    │ a₃₁·b₁₃ + a₃₂·b₂₃ │
└                  ┘                              └                   ┘
      (3, 2)                                               (3,)
```

Los resultados son idénticos a la diagonal de $A \cdot B$ de arriba: mismos subíndices, mismas sumas. Pero nunca se construyó la matriz $3 \times 3$.

La memoria pasa de $(n, n) = (3, 3) = 9$ elementos a $(n, p) = (3, 2) = 6$ elementos.

6. Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`.

In [214]:
# EfficientQDA usa el truco de diag(A·B) = sum(A ⊙ B^T, axis=1)
# para evitar construir la matriz n×n.
#
# En FasterQDA el cuello de botella era:
#   quad = U^T @ SU  → (k, n, n)   ← matriz n×n por clase
#   mahal = diagonal(quad)          ← se descarta casi todo
#
# Aplicando la propiedad con A = U^T (k,n,p) y B = SU (k,p,n):
#   diag(U^T @ SU) = sum(U^T ⊙ SU^T, axis=1) = sum(U^T ⊙ SU^T, axis=2)
# Equivalentemente, trabajando en (k,p,n):
#   diag = sum(U * SU, axis=1)  → (k, n)  directamente
#
# Nunca se construye la matriz (k, n, n). Memoria: O(np) en vez de O(n²).

class EfficientQDA(TensorizedQDA):

    def predict(self, X):
        # X: (p, n), tensor_means: (k, p, 1), tensor_inv_cov: (k, p, p)

        # Centrar: (p, n) - (k, p, 1) → (k, p, n)
        U = X - self.tensor_means

        # Producto Σ_j⁻¹ @ U: (k, p, p) @ (k, p, n) → (k, p, n)
        SU = self.tensor_inv_cov @ U

        # Truco Hadamard: diag(Uᵀ @ SU) = sum(U * SU, axis=1) → (k, n)
        # U y SU tienen shape (k, p, n), el * es element-wise,
        # y sum(axis=1) colapsa la dimensión p → producto punto por observación
        mahal = np.sum(U * SU, axis=1)  # (k, n)

        # Log-posterior: (k, 1) - (k, n) → (k, n)
        log_det = 0.5 * np.log(LA.det(self.tensor_inv_cov))
        log_post = (self.log_a_priori + log_det).reshape(-1, 1) - 0.5 * mahal

        # Clase ganadora por observación: (n,) → (1, n)
        return np.argmax(log_post, axis=0).reshape(1, -1)

7. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

In [215]:
# Benchmark de las 4 variantes de QDA hasta ahora
# QDA: base, for-loop sobre clases y observaciones
# TensorizedQDA: tensoriza sobre clases, for-loop sobre observaciones
# FasterQDA: tensoriza sobre ambas, pero construye matriz n×n
# EfficientQDA: tensoriza sobre ambas, evita la matriz n×n con Hadamard

X_full, y_full = get_wine_dataset()
y_full_encoded = label_encode(y_full)

b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 500,
    warmup = 50,
    mem_runs = 50,
    test_sz = 0.3,
    same_splits = False
)

to_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA]

for model in to_bench:
    b.bench(model)

summ = b.summary(baseline='QDA')
summ[[
    'train_median_ms', 'test_median_ms', 'mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

Benching params:
Total runs: 600
Warmup runs: 50
Peak Memory usage runs: 50
Running time runs: 500
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


QDA (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/500 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/500 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/500 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/500 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.121375,1.247729,0.984148,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.122333,0.613854,0.984333,0.992165,2.032615,0.999974,0.633286
FasterQDA,0.112916,0.032166,0.982259,1.074914,38.790305,0.999974,0.067698
EfficientQDA,0.111396,0.027208,0.983926,1.089586,45.858902,1.005635,0.102742


#### Análisis de resultados

**Tiempos de entrenamiento:** las 4 variantes tienen tiempos de train prácticamente iguales (~0.11-0.12 ms, speedup ~1x). Esto tiene sentido porque todas usan el mismo `_fit_params` de QDA — la diferencia entre ellas está solo en cómo predicen, no en cómo entrenan.

**Tiempos de predicción (test):** acá es donde se ven las mejoras reales:

- **QDA** (1.25 ms): la más lenta, tiene doble for-loop (clases × observaciones).
- **TensorizedQDA** (0.61 ms, ~2x speedup): elimina el for-loop sobre clases tensorizando con `np.stack`. La mejora es moderada porque el for sobre observaciones sigue.
- **FasterQDA** (0.03 ms, ~39x speedup): elimina ambos for-loops haciendo batch sobre todas las observaciones. El salto es enorme porque pasar de un loop de Python a operaciones matriciales en BLAS es mucho más rápido, incluso construyendo la matriz $n \times n$ de más.
- **EfficientQDA** (0.03 ms, ~46x speedup): evita la matriz $n \times n$ con el truco de Hadamard. Es ligeramente más rápido que FasterQDA, aunque con el dataset Wine ($n \approx 50$ en test) la diferencia es chica porque la matriz $n \times n$ es pequeña.

**Memoria (test):**
- FasterQDA usa ~7% de la memoria de QDA, y EfficientQDA ~10%. Ambos son mucho más eficientes que el baseline. La diferencia entre FasterQDA y EfficientQDA en memoria podría ser más visible con datasets más grandes donde la matriz $n \times n$ pese más.

**Accuracy:** las 4 variantes tienen accuracy prácticamente idéntico (~98.2-98.4%), lo cual confirma que todas calculan lo mismo — las diferencias son solo de implementación, no de modelo.

**¿Se condice con lo esperado?** Sí. El mayor salto es al eliminar los for-loops de Python (FasterQDA), no al optimizar la memoria (EfficientQDA). Esto es esperable con un dataset chico como Wine. Con datasets más grandes (ej: Letters, $n \approx 16000$) la ventaja de EfficientQDA debería ser más notoria tanto en tiempo como en memoria.

In [218]:
# Benchmark con Letters dataset (n ≈ 20000, k = 26, p = 16)
# Con un dataset más grande se debería notar más la diferencia
# entre FasterQDA y EfficientQDA (la matriz n×n pesa mucho más)

X_letter, y_letter = get_letters_dataset()
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

b_letter = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

to_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA]

for model in to_bench:
    b_letter.bench(model)

summ_letter = b_letter.summary(baseline='QDA')
summ_letter[[
    'train_median_ms', 'test_median_ms', 'mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,5.296750,714.364583,0.886117,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,5.233292,152.329895,0.885303,1.012126,4.689589,1.001817,0.631235
FasterQDA,5.251750,72.927271,0.884827,1.008569,9.795575,1.000000,0.000030
EfficientQDA,4.851166,4.786646,0.884890,1.091851,149.241141,0.998189,0.002498


#### Análisis de resultados con Letters

Con un dataset grande ($n \approx 20000$, $k = 26$, $p = 16$) las diferencias se amplifican enormemente:

**Tiempos de entrenamiento:** siguen siendo similares (~5 ms, speedup ~1x).

**Tiempos de predicción (test):**

- **QDA** (714 ms): el doble for-loop escala muy mal con $n$ grande.
- **TensorizedQDA** (152 ms, ~4.7x): mejora moderada, pero el for sobre observaciones sigue siendo costoso con $n \approx 4000$ en test.
- **FasterQDA** (73 ms, ~10x): elimina ambos for-loops pero ahora la matriz $n \times n$ ($4000 \times 4000$) pesa — se nota que el speedup es mucho menor que con Wine (10x vs 39x). La construcción de esa matriz frena la ganancia.
- **EfficientQDA** (4.8 ms, **~149x** speedup): acá se ve el impacto real del truco de Hadamard. Al evitar la matriz $n \times n$, es **15 veces más rápido que FasterQDA**. Con Wine la diferencia era despreciable; con Letters es brutal.

**Memoria (test):**
- FasterQDA usa ~0.003% de la memoria de QDA — pero esto es engañoso, la `test_mem_reduction` tan baja indica un pico de memoria enorme por la matriz $(k, n, n)$. Con $k=26$ y $n \approx 4000$, eso son 26 matrices de $4000 \times 4000$.
- EfficientQDA usa ~0.25% de la memoria de QDA, mucho más eficiente que FasterQDA porque nunca construye esas matrices.

**Accuracy:** las 4 variantes dan ~88.5%, confirmando que son el mismo modelo.

**Conclusión:** con datos chicos (Wine) la diferencia entre FasterQDA y EfficientQDA era marginal. Con datos grandes (Letters) se confirma lo que anticipábamos: evitar la matriz $n \times n$ pasa de ser una optimización teórica a una necesidad práctica.

## Cholesky

Hasta ahora todos los esfuerzos fueron enfocados en realizar una predicción más rápida. Los tiempos de entrenamiento (teóricos al menos) siguen siendo los mismos o hasta (minúsculamente) peores, dado que todas las mejoras siguen llamando al método `_fit_params` original de `QDA`.

La descomposición/factorización de [Cholesky](https://en.wikipedia.org/wiki/Cholesky_decomposition#Statement) permite factorizar una matriz definida positiva $A = LL^T$ donde $L$ es una matriz triangular inferior. En particular, si bien se asume que $p \ll n$, invertir la matriz de covarianzas $\Sigma$ para cada clase impone un cuello de botella que podría alivianarse. Teniendo en cuenta que las matrices de covarianza son simétricas y salvo degeneración, definidas positivas, Cholesky como mínimo debería permitir invertir la matriz más rápido.

*Nota: observar que calcular* $A^{-1}b$ *equivale a resolver el sistema* $Ax=b$.

### 3) Diferencias entre implementaciones de `QDA_Chol`

7. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.

#### Diferencias entre QDA y QDA_Chol1

QDA trabaja directamente con la inversa de la covarianza $\Sigma_j^{-1}$: calcula `np.cov`, la invierte con `LA.inv`, y en la predicción hace la multiplicación matricial completa $(x-\mu_j)^T \Sigma_j^{-1} (x-\mu_j)$.

QDA_Chol1 en cambio descompone la covarianza con Cholesky: $\Sigma_j = L_j L_j^T$, e invierte $L_j$ con `LA.inv`. En la predicción hace algo distinto: calcula $y = L_j^{-1}(x - \mu_j)$ y después simplemente hace `(y**2).sum()`.

Esto funciona porque:

$$
y^T y = (L_j^{-1}(x-\mu_j))^T (L_j^{-1}(x-\mu_j)) = (x-\mu_j)^T \underbrace{(L_j^{-1})^T L_j^{-1}}_{= \Sigma_j^{-1}} (x-\mu_j)
$$

O sea, $\|y\|^2$ es exactamente la distancia de Mahalanobis, pero en vez de hacer una multiplicación matricial $(1,p) \times (p,p) \times (p,1)$, se reduce a sumar los cuadrados de un vector. Es más barato.

Para el determinante también hay una simplificación. Como $L_j^{-1}$ es triangular, su determinante es el producto de la diagonal:

$$
\frac{1}{2}\log|\Sigma_j^{-1}| = \log\prod_i (L_j^{-1})_{ii}
$$

En el código esto es `np.log(L_inv.diagonal().prod())` — no necesita calcular `LA.det` de una matriz $p \times p$.

En resumen, Cholesky simplifica tanto la distancia de Mahalanobis (norma de un vector en vez de forma cuadrática) como el determinante (producto de diagonal en vez de `det`). 

8. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?

Si $A = LL^T$, entonces invertimos ambos lados:

$$
A^{-1} = (LL^T)^{-1} = (L^{-1})^T L^{-1}
$$

Porque la inversa de un producto se invierte en orden: $(AB)^{-1} = B^{-1}A^{-1}$.

En la forma cuadrática de QDA necesitamos calcular:

$$
(x - \mu)^T \Sigma^{-1} (x - \mu) = (x - \mu)^T (L^{-1})^T L^{-1} (x - \mu)
$$

Si defino $y = L^{-1}(x - \mu)$, entonces:

$$
(x - \mu)^T (L^{-1})^T L^{-1} (x - \mu) = y^T y = \|y\|^2 = \sum_i y_i^2
$$

La forma cuadrática se reduce a calcular la norma al cuadrado de un vector. Esto es útil porque:

- Calcular $y = L^{-1}(x - \mu)$ es resolver el sistema triangular $Ly = (x - \mu)$, que se puede hacer con forward substitution en $O(p^2)$ sin necesidad de invertir $L$ explícitamente (como hace `QDA_Chol2` con `solve_triangular`).
- $\|y\|^2$ es simplemente sumar cuadrados, mucho más barato que la multiplicación matricial $(1,p) \times (p,p) \times (p,1)$ que hace QDA.
- Nunca hace falta calcular ni almacenar $\Sigma^{-1}$ completa.

Esto es exactamente lo que implementan las tres variantes Cholesky del código.

8. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?

Las tres hacen lo mismo matemáticamente: descomponen $\Sigma_j = L_j L_j^T$ y usan eso para calcular la distancia de Mahalanobis como $\|y\|^2$ donde $y = L_j^{-1}(x - \mu_j)$. La diferencia está en **cómo obtienen $y$**:

**QDA_Chol1**: calcula $L_j^{-1}$ explícitamente con `LA.inv` (inversión genérica) y guarda $L_j^{-1}$. En la predicción hace `y = L_inv @ unbiased_x` (multiplicación matriz-vector). El problema es que `LA.inv` no aprovecha que $L_j$ es triangular inferior — usa el mismo algoritmo que para cualquier matriz $p \times p$.

**QDA_Chol2**: no invierte $L_j$. Guarda $L_j$ directamente y en la predicción resuelve el sistema $L_j y = (x - \mu_j)$ con `solve_triangular` (forward substitution). Esto es más eficiente que invertir porque la forward substitution explota directamente la estructura triangular: resuelve de arriba hacia abajo, cada variable depende solo de las anteriores. Además, no necesita almacenar $L_j^{-1}$.

**QDA_Chol3**: como Chol1, calcula y guarda $L_j^{-1}$, pero usa `dtrtri` en vez de `LA.inv`. `dtrtri` es una rutina de LAPACK especializada para invertir matrices triangulares — aprovecha la estructura triangular y es más rápida y numéricamente estable que `LA.inv`.

El determinante se calcula igual en las tres: como $L$ (o $L^{-1}$) es triangular, $|L| = \prod_i L_{ii}$. Chol1 y Chol3 usan `log(L_inv.diagonal().prod())`, Chol2 usa `-log(L.diagonal().prod())` (signo negativo porque trabaja con $L$ en vez de $L^{-1}$).

En resumen:

| | Guarda | Cómo obtiene $y$ | Inversión |
|---|---|---|---|
| **Chol1** | $L_j^{-1}$ | `L_inv @ x` (matmul) | `LA.inv` (genérica) |
| **Chol2** | $L_j$ | `solve_triangular(L, x)` (forward sub.) | No invierte |
| **Chol3** | $L_j^{-1}$ | `L_inv @ x` (matmul) | `dtrtri` (triangular, LAPACK) |

9. Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?

In [219]:
# Benchmark de las 7 variantes con Wine dataset
X_full, y_full = get_wine_dataset()
y_full_encoded = label_encode(y_full)

b_wine_7 = Benchmark(
    X_full, y_full_encoded,
    n_runs = 500,
    warmup = 50,
    mem_runs = 50,
    test_sz = 0.3,
    same_splits = False
)

to_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA, QDA_Chol1, QDA_Chol2, QDA_Chol3]

for model in to_bench:
    b_wine_7.bench(model)

summ_wine_7 = b_wine_7.summary(baseline='QDA')
summ_wine_7[[
    'train_median_ms', 'test_median_ms', 'mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

Benching params:
Total runs: 600
Warmup runs: 50
Peak Memory usage runs: 50
Running time runs: 500
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


QDA (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/500 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/500 [00:00<?, ?it/s]

...

QDA_Chol2 (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/500 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/500 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.118209,1.250875,0.984148,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.125104,0.617542,0.984333,0.944882,2.025570,1.000000,0.625938
FasterQDA,0.111562,0.031667,0.982259,1.059572,39.500883,1.005714,0.066912
EfficientQDA,0.111000,0.026667,0.983926,1.064941,46.907208,1.000000,0.101550
QDA_Chol1,0.146459,0.676521,0.984963,0.807110,1.848981,1.004714,0.946678
QDA_Chol2,0.128604,1.870521,0.984630,0.919167,0.668731,0.991165,0.909990
QDA_Chol3,0.124479,0.674312,0.984519,0.949626,1.855037,1.006215,0.975512


#### Análisis Wine (7 variantes)

Las variantes Cholesky:

- **Chol1** (0.68 ms test, ~1.8x speedup): ligeramente más rápido que QDA base. El train es más lento (0.15 ms vs 0.12 ms) porque tiene que hacer Cholesky + `LA.inv` en vez de solo `LA.inv`.
- **Chol2** (1.87 ms test, **0.67x** — más lento que QDA): llama la atención que es la peor. `solve_triangular` se llama una vez por cada observación dentro del for-loop, y ese overhead por llamada se acumula. Multiplicar `L_inv @ x` directamente (como hacen Chol1 y Chol3) es más rápido cuando ya tenés la inversa precalculada.
- **Chol3** (0.67 ms test, ~1.9x speedup): prácticamente igual a Chol1. Teóricamente debería ser mejor porque `dtrtri` aprovecha que $L$ es triangular, pero con $p = 13$ la matriz es tan chica que no se nota.

**¿Alguna claramente mejor?** Chol1 y Chol3 empatan, ambas mejoran levemente a QDA en predicción.

**¿Alguna claramente peor?** Sí, Chol2 es la única variante más lenta que QDA base.

En definitiva, ninguna Cholesky compite con FasterQDA o EfficientQDA en velocidad de predicción, porque siguen usando el for-loop sobre observaciones. La ventaja de Cholesky pasa por otro lado: estabilidad numérica y servir como base para tensorizar.

In [220]:
# Benchmark de las 7 variantes con Letters dataset
X_letter, y_letter = get_letters_dataset()
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

b_letter_7 = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

to_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA, QDA_Chol1, QDA_Chol2, QDA_Chol3]

for model in to_bench:
    b_letter_7.bench(model)

summ_letter_7 = b_letter_7.summary(baseline='QDA')
summ_letter_7[[
    'train_median_ms', 'test_median_ms', 'mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

...

QDA_Chol2 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,5.146396,717.776437,0.886117,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,5.205688,152.566167,0.885303,0.988610,4.704689,1.001817,0.631671
FasterQDA,5.121645,72.371062,0.884827,1.004833,9.918003,1.000000,0.000030
EfficientQDA,4.976167,4.842396,0.884890,1.034209,148.227538,0.998189,0.002500
QDA_Chol1,5.451979,374.777167,0.884770,0.943950,1.915209,1.000358,1.025147
QDA_Chol2,5.105459,1070.494605,0.885433,1.008018,0.670509,1.000046,1.020481
QDA_Chol3,5.187938,371.909125,0.885807,0.991993,1.929978,1.000482,1.027538


#### Análisis Letters (7 variantes)

Con Letters se amplifica todo lo que vimos con Wine. EfficientQDA sigue dominando (~148x speedup, 4.8 ms), y FasterQDA baja a ~10x por la matriz $n \times n$.

Las Cholesky:

- **Chol1** (375 ms test, ~1.9x speedup): mejora modesta sobre QDA (718 ms). El for-loop sigue siendo el cuello de botella.
- **Chol2** (1070 ms test, **0.67x** — más lento que QDA): de nuevo la peor. Con $n \approx 4000$ en test y $k = 26$ clases, son ~100.000 llamadas a `solve_triangular` — el overhead se vuelve muy pesado.
- **Chol3** (372 ms test, ~1.9x speedup): prácticamente igual a Chol1. Con $p = 16$ la diferencia entre `dtrtri` y `LA.inv` sigue siendo mínima.

**¿Alguna claramente mejor?** Entre las Cholesky, Chol1 y Chol3 empatan con ~1.9x speedup, lo cual es muy modesto comparado con EfficientQDA (148x).

**¿Alguna claramente peor?** Chol2 es claramente la peor, tanto con Wine como con Letters. El patrón es consistente.

La conclusión que se saca es que las variantes Cholesky sin tensorizar no aportan mucho en velocidad de predicción. Su valor real está en la estabilidad numérica (mejor condicionamiento, como discutimos antes) y en que son la base natural para implementar una versión tensorizada que sí podría competir con EfficientQDA.

### 4) Optimización

12. Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones según corresponda. Se recomienda heredarlo de alguna de las implementaciones de `QDA_Chol`, aunque la elección de cuál de ellas queda a cargo del alumno según lo observado en los benchmarks de puntos anteriores.

In [233]:
# TensorizedChol hereda de QDA_Chol3 porque:
# - Chol1 y Chol3 tuvieron performance similar y ambas superaron a Chol2
# - Chol3 usa dtrtri (LAPACK, especializada para triangulares) que es
#   teóricamente mejor que LA.inv (genérica) de Chol1
#
# Como TensorizedQDA, solo tensoriza sobre las k clases.
# El for-loop sobre observaciones se mantiene (heredado de BaseBayesianClassifier).

class TensorizedChol(QDA_Chol3):

    def _fit_params(self, X, y):
        super()._fit_params(X, y)

        # Stack en tensores para operar sobre todas las clases a la vez
        self.tensor_L_inv = np.stack(self.L_invs)    # (k, p, p)
        self.tensor_means = np.stack(self.means)      # (k, p, 1)

        # Precomputar log-det: constante por clase, evita recalcularlo n×k veces
        self.log_dets = np.array([
            np.log(L_inv.diagonal().prod()) for L_inv in self.L_invs
        ])  # (k,)

    def _predict_log_conditionals(self, x):
        # Calcula log f_j(x) para las k clases a la vez (una sola observación)
        unbiased_x = x - self.tensor_means            # (k, p, 1)
        Y = self.tensor_L_inv @ unbiased_x            # (k, p, 1)
        return self.log_dets - 0.5 * (Y ** 2).sum(axis=1).flatten()  # (k,)

    def _predict_one(self, x):
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

13. Implementar el modelo `EfficientChol` combinando los insights de `EfficientQDA` y `TensorizedChol`. Si se desea, se puede implementar `FasterChol` como ayuda, pero no se contempla para el punto.

#### EfficientChol: combinando EfficientQDA y Cholesky

La idea es juntar los dos insights que fuimos desarrollando:

En EfficientQDA, para evitar la matriz $n \times n$, usamos el truco de Hadamard:

$$\text{diag}(U^T \Sigma^{-1} U) = \text{sum}(U \odot (\Sigma^{-1} U), \text{axis}=1)$$

Con Cholesky, $\Sigma^{-1} = (L^{-1})^T L^{-1}$, entonces la forma cuadrática se reescribe como:

$$\text{diag}(U^T (L^{-1})^T L^{-1} U) = \text{diag}((L^{-1}U)^T (L^{-1}U)) = \text{diag}(Y^T Y)$$

donde $Y = L^{-1}U$. Y aplicando Hadamard a $\text{diag}(Y^T Y)$:

$$\text{diag}(Y^T Y) = \text{sum}(Y \odot Y, \text{axis}=1) = \text{sum}(Y^2, \text{axis}=1)$$

El resultado es que con Cholesky el truco de Hadamard se simplifica a **sumar cuadrados**. En EfficientQDA necesitaba multiplicar dos matrices distintas (`U * SU`), acá alcanza con `Y²` porque $Y$ ya "absorbe" la inversa.

In [234]:
# EfficientChol combina:
# - De EfficientQDA: tensorizar sobre clases Y observaciones, evitando la matriz n×n
# - De Cholesky: la descomposición que convierte la forma cuadrática en una norma
#
# A diferencia de TensorizedChol (que solo tensoriza sobre clases y mantiene
# el for-loop sobre observaciones), EfficientChol elimina ambos for-loops.
#
# La conexión clave: en EfficientQDA usábamos el truco de Hadamard
#   diag(Uᵀ Σ⁻¹ U) = sum(U * (Σ⁻¹ U), axis=1)
#
# Con Cholesky, Σ⁻¹ = (L⁻¹)ᵀ L⁻¹, entonces:
#   diag(Uᵀ (L⁻¹)ᵀ L⁻¹ U) = diag(Yᵀ Y)   donde Y = L⁻¹U
#
# Y el Hadamard de diag(Yᵀ Y) se simplifica a:
#   sum(Y², axis=1)
#
# Con Cholesky, el truco de Hadamard se reduce a sumar cuadrados.
# No hace falta multiplicar U * SU con dos matrices distintas como en
# EfficientQDA — alcanza con Y² porque Y ya "absorbe" la inversa.

class EfficientChol(TensorizedChol):
    # Hereda de TensorizedChol (que ya tiene _fit_params con tensores y log_dets)
    # Solo sobreescribe predict para eliminar el for-loop sobre observaciones

    def predict(self, X):
        # X: (p, n) — todas las observaciones de golpe

        # Centrar: (p, n) - (k, p, 1) → (k, p, n)
        U = X - self.tensor_means

        # Y = L⁻¹ U: (k, p, p) @ (k, p, n) → (k, p, n)
        Y = self.tensor_L_inv @ U

        # Hadamard: diag(Yᵀ Y) = sum(Y², axis=1) → (k, n)
        # Equivalente al sum(U * SU, axis=1) de EfficientQDA,
        # pero más simple porque Y ya contiene L⁻¹
        mahal = (Y ** 2).sum(axis=1)  # (k, n)

        # Log-posterior: (k, 1) - (k, n) → (k, n)
        log_post = (self.log_a_priori + self.log_dets).reshape(-1, 1) - 0.5 * mahal

        # Clase ganadora: (n,) → (1, n)
        return np.argmax(log_post, axis=0).reshape(1, -1)

13. Comparar la performance de las 9 variantes de QDA implementadas ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

In [235]:
# Benchmark de las 9 variantes con Wine dataset
X_full, y_full = get_wine_dataset()
y_full_encoded = label_encode(y_full)

b_wine_9 = Benchmark(
    X_full, y_full_encoded,
    n_runs = 500,
    warmup = 50,
    mem_runs = 50,
    test_sz = 0.3,
    same_splits = False
)

to_bench = [
    QDA, TensorizedQDA, FasterQDA, EfficientQDA,
    QDA_Chol1, QDA_Chol2, QDA_Chol3,
    TensorizedChol, EfficientChol
]

for model in to_bench:
    b_wine_9.bench(model)

summ_wine_9 = b_wine_9.summary(baseline='QDA')
summ_wine_9[[
    'train_median_ms', 'test_median_ms', 'mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

Benching params:
Total runs: 600
Warmup runs: 50
Peak Memory usage runs: 50
Running time runs: 500
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


QDA (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/500 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/500 [00:00<?, ?it/s]

...

TensorizedChol (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/500 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/50 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/500 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.119979,1.242312,0.984148,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.121000,0.607479,0.984333,0.991566,2.045030,1.000000,0.633286
FasterQDA,0.111417,0.032208,0.982259,1.076851,38.571549,1.008198,0.067698
EfficientQDA,0.112417,0.027458,0.983926,1.067272,45.243276,1.005662,0.102742
QDA_Chol1,0.155645,0.690624,0.984963,0.770851,1.798825,1.004714,0.957791
QDA_Chol2,0.127667,1.844229,0.984630,0.939785,0.673622,0.988382,0.933080
QDA_Chol3,0.124625,0.672833,0.984519,0.962724,1.846391,1.017071,0.984804
TensorizedChol,0.131854,0.300521,0.984037,0.909942,4.133862,1.008729,0.610291
EfficientChol,0.127480,0.016584,0.984111,0.941167,74.910292,1.017071,0.127976


#### Análisis Wine (9 variantes)

Las variantes sin Cholesky se comportan como antes. Lo nuevo son TensorizedChol y EfficientChol:

- **TensorizedChol** (0.30 ms test, ~4x speedup): tensoriza sobre clases pero mantiene el for-loop sobre observaciones, igual que TensorizedQDA. El speedup es modesto pero un poco mejor que TensorizedQDA (~2x) por la simplificación de Cholesky.
- **EfficientChol** (0.017 ms test, **~75x** speedup): al eliminar ambos for-loops, es la más rápida de todas. Supera incluso a EfficientQDA (~45x) porque `(Y**2).sum()` (sumar cuadrados) es más simple que `sum(U * SU)` (Hadamard con dos matrices distintas).

La diferencia entre TensorizedChol y EfficientChol (~4x vs ~75x) muestra claramente el impacto de eliminar el for-loop sobre observaciones.

**¿Se condice con lo esperado?** Sí. TensorizedChol se comporta como TensorizedQDA (mejora moderada), y EfficientChol es la ganadora. El train es ligeramente más lento para las Cholesky (~0.13 ms vs ~0.11 ms) por la descomposición + inversión triangular, pero se compensa de sobra en test.

In [236]:
# Benchmark de las 9 variantes con Letters dataset
X_letter, y_letter = get_letters_dataset()
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

b_letter_9 = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

to_bench = [
    QDA, TensorizedQDA, FasterQDA, EfficientQDA,
    QDA_Chol1, QDA_Chol2, QDA_Chol3,
    TensorizedChol, EfficientChol
]

for model in to_bench:
    b_letter_9.bench(model)

summ_letter_9 = b_letter_9.summary(baseline='QDA')
summ_letter_9[[
    'train_median_ms', 'test_median_ms', 'mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

...

TensorizedChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,5.046937,706.139229,0.886117,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,5.040458,150.786874,0.885303,1.001285,4.683028,1.001817,0.631816
FasterQDA,4.991813,70.897875,0.884827,1.011043,9.959949,1.000000,0.000030
EfficientQDA,4.736292,5.511917,0.884890,1.065588,128.111368,0.998189,0.002500
QDA_Chol1,5.241063,370.539104,0.884770,0.962961,1.905708,1.000975,1.024910
QDA_Chol2,5.003562,1050.622687,0.885433,1.008669,0.672115,1.000291,1.023064
QDA_Chol3,5.081333,367.222312,0.885807,0.993231,1.922920,1.000645,1.028488
TensorizedChol,4.910042,28.446271,0.884995,1.027881,24.823613,1.000727,0.617074
EfficientChol,4.767209,4.084208,0.885720,1.058678,172.895021,0.999610,0.002500


#### Análisis Letters (9 variantes)

Con Letters las diferencias se amplifican:

- **TensorizedChol** (28.4 ms test, ~25x speedup): tensoriza sobre clases pero el for-loop sobre observaciones con $n \approx 4000$ lo limita.
- **EfficientChol** (4.08 ms test, **~173x** speedup): la más rápida de todas, superando a EfficientQDA (~128x). La simplificación de Cholesky (`(Y**2).sum()` vs `sum(U * SU)`) tiene impacto real.

**¿Se condice con lo esperado?** Sí:

- **EfficientChol es la ganadora absoluta** en predicción (~173x speedup)
- La combinación de Cholesky + tensorización completa da lo mejor de ambos mundos: estabilidad numérica (Cholesky) + velocidad (sin for-loops, sin matriz $n \times n$)
- La diferencia entre TensorizedChol (~25x) y EfficientChol (~173x) confirma que eliminar el for-loop sobre observaciones es el paso más importante

**Accuracy:** las 9 variantes dan ~88.5%, confirmando que todas son el mismo modelo con distinta implementación.

## Importante:

Las métricas que se observan al realizar benchmarking son muy dependientes del código que se ejecuta, y por tanto de las versiones de las librerías utilizadas. Una forma de unificar esto es utilizando un gestor de versiones y paquetes como _uv_ o _Poetry_, otra es simplemente usando una misma VM como la que provee Colab.

**Cada equipo debe informar las versiones de Python, NumPy y SciPy con que fueron obtenidos los resultados. En caso de que sean múltiples, agregar todos los casos**. La siguiente celda provee una ayuda para hacerlo desde un notebook, aunque como es una secuencia de comandos también sirve para consola.

In [237]:
import sys
import numpy
import scipy

print(f"Python {sys.version}")
print(f"NumPy  {numpy.__version__}")
print(f"SciPy  {scipy.__version__}")

Python 3.12.12 (main, Dec 17 2025, 21:07:08) [Clang 21.1.4 ]
NumPy  2.4.2
SciPy  1.17.1


**Comentario:** yo utilicé los siguientes parámetros para mi run de prueba. Esto NO significa que ustedes tengan que usar los mismos, tampoco el mismo dataset. Se agregó al notebook simplemente porque fue una pregunta común en cohortes anteriores.

In [238]:
# dataset de letters
X_letter, y_letter = get_letters_dataset()

# encoding de labels
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

# instanciacion del benchmark
b = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2
